# Figures du rapport

Chaque figure est lue dans les fichiers de `data/processed/` et écrite dans `figures/`. Aucune n'est calculée ici : ce carnet ne fait que dessiner des résultats déjà produits et contrôlés.

Couleurs : bleu pour le thème, orange pour le témoin, gris pour le marché. Les épisodes de tension sont en fond gris clair.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

RACINE = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / ".git").exists())
DONNEES = RACINE / "data" / "processed"
FIGURES = RACINE / "figures"
FIGURES.mkdir(exist_ok=True)

THEME, TEMOIN, AQUA, MARCHE = "#2a78d6", "#eb6834", "#1baf7a", "#898781"
ENCRE, ENCRE_2, GRILLE, FOND = "#0b0b0b", "#52514e", "#e1e0d9", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": FOND, "axes.facecolor": FOND, "savefig.facecolor": FOND,
    "font.size": 11, "axes.titlesize": 14, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlepad": 24,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": ENCRE_2, "xtick.color": ENCRE_2, "ytick.color": ENCRE_2,
    "axes.grid": True, "grid.color": GRILLE, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
})

valeurs = pd.read_csv(DONNEES / "valeurs_portefeuilles.csv", index_col=0, parse_dates=True)
temoin = pd.read_csv(DONNEES / "temoin_valeurs.csv", index_col=0, parse_dates=True)
marche = pd.read_csv(DONNEES / "valeurs_comparaisons.csv", index_col=0, parse_dates=True)["SPY"]
episodes = pd.read_csv(DONNEES / "episodes_tension.csv", parse_dates=["debut", "creux"])


def tensions(ax):
    for e in episodes.itertuples():
        ax.axvspan(e.debut, e.creux, color="#f0efec", zorder=0, lw=0)


def etiquette(ax, serie, texte, couleur, decalage=0):
    x, y = serie.dropna().index[-1], serie.dropna().iloc[-1]
    ax.annotate(texte, (x, y), xytext=(6, decalage), textcoords="offset points",
                color=ENCRE, fontsize=10, va="center",
                bbox=dict(boxstyle="round,pad=0.2", fc=FOND, ec=couleur, lw=1.5))


def source(fig, texte):
    fig.text(0.01, -0.03, texte, fontsize=8.5, color=ENCRE_2)


def enregistrer(fig, nom):
    fig.savefig(FIGURES / nom, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print("ecrit", nom)

## 1. Le thème, son témoin et le marché

In [2]:
fig, ax = plt.subplots(figsize=(10, 5.4))
tensions(ax)
series = [(valeurs["P1_reeq"], "P1 thème IA", THEME, 0),
          (temoin["T1S_reeq"], "T1S témoin, mêmes secteurs", TEMOIN, -12),
          (marche, "SPY marché", MARCHE, 0)]
for s, nom, couleur, dy in series:
    ax.plot(s.index, s, color=couleur)
    etiquette(ax, s, nom, couleur, dy)
ax.set_yscale("log")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", " ")))
ax.set_ylabel("Valeur, base 100 en janvier 2000 (échelle log)")
ax.set_title("Le thème fait mieux que le marché, mais le témoin aussi")
ax.text(0, 1.01, "L'essentiel de l'écart avec SPY vient du biais de survie commun aux deux groupes, pas de l'IA",
        transform=ax.transAxes, color=ENCRE_2, fontsize=10)
ax.legend([plt.Line2D([], [], color=c) for _, _, c, _ in series], [n for _, n, _, _ in series],
          loc="upper left", frameon=False)
source(fig, "Portefeuilles équipondérés rééquilibrés en janvier, frais de 10 pb. Fond gris : épisodes de tension de SPY.")
enregistrer(fig, "1_valeurs.png")

ecrit 1_valeurs.png


## 2. Les replis

In [3]:
fig, ax = plt.subplots(figsize=(10, 4.6))
for s, nom, couleur, dy in series:
    repli = s.dropna() / s.dropna().cummax() - 1
    ax.plot(repli.index, 100 * repli, color=couleur, lw=1.6, label=nom)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f} %"))
ax.set_ylabel("Écart au plus haut précédent")
ax.set_title("Trois séries, les mêmes crises")
ax.text(0, 1.01, "Replis maximaux voisins : -52 % pour le thème, -50 % pour le témoin, -55 % pour SPY",
        transform=ax.transAxes, color=ENCRE_2, fontsize=10)
ax.legend(loc="lower right", frameon=False)
source(fig, "Source : data/processed/valeurs_portefeuilles.csv, temoin_valeurs.csv, valeurs_comparaisons.csv.")
enregistrer(fig, "2_replis.png")

ecrit 2_replis.png


## 3. Qui porte le risque de P1

In [4]:
maillons = pd.read_csv(DONNEES / "risque_par_maillon.csv")
m = maillons[maillons.serie == "P1_reeq"].sort_values("risque")
noms = {"vendeurs": "Vendeurs de puces et systèmes", "amont des puces": "Amont des semi-conducteurs",
        "equipement": "Équipement", "acheteurs": "Acheteurs d'infrastructure",
        "electricite": "Électricité", "immobilier": "Immobilier", "matieres et energie": "Matières et énergie"}
y = np.arange(len(m))
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.barh(y + 0.2, 100 * m.poids, height=0.38, color=MARCHE, label="Part de l'argent")
ax.barh(y - 0.2, 100 * m.risque, height=0.38, color=THEME, label="Part du risque")
for i, (p, r) in enumerate(zip(m.poids, m.risque)):
    ax.text(100 * p + 0.8, i + 0.2, f"{100 * p:.0f} %", va="center", fontsize=9, color=ENCRE_2)
    ax.text(100 * r + 0.8, i - 0.2, f"{100 * r:.0f} %", va="center", fontsize=9, color=ENCRE)
ax.set_yticks(y, [noms.get(n, n) for n in m.maillon])
ax.grid(axis="y", visible=False)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f} %"))
ax.set_title("Les puces portent l'essentiel du risque de P1")
ax.text(0, 1.01, "Contribution de chaque maillon à la volatilité, covariance des 252 dernières séances",
        transform=ax.transAxes, color=ENCRE_2, fontsize=10)
ax.legend(loc="lower right", frameon=False)
source(fig, "P1 rééquilibré au 4 septembre 2026. Source : data/processed/risque_par_maillon.csv.")
enregistrer(fig, "3_risque_par_maillon.png")

ecrit 3_risque_par_maillon.png


## 4. La concentration qui se forme sans rééquilibrage

In [5]:
evolution = pd.read_csv(DONNEES / "risque_concentration_evolution.csv", parse_dates=["date"])
fig, ax = plt.subplots(figsize=(10, 4.6))
for serie, nom, couleur, dy in [("P1_reeq", "Rééquilibré chaque janvier", THEME, 0),
                                ("P1_cons", "Conservé sans rééquilibrage", TEMOIN, 0)]:
    s = evolution[evolution.serie == serie].set_index("date").n_eff
    ax.plot(s.index, s, color=couleur)
    etiquette(ax, s, f"{nom} : {s.iloc[-1]:.1f}", couleur, dy)
ax.set_ylim(0, None)
ax.set_ylabel("Nombre effectif de lignes (1 / Herfindahl)")
ax.set_title("Sans rééquilibrage, P1 se concentre en six ans")
ax.text(0, 1.01, "Le même univers de 134 entreprises ; une opération par an sépare les deux courbes",
        transform=ax.transAxes, color=ENCRE_2, fontsize=10)
source(fig, "Relevés mensuels des poids. Source : data/processed/risque_concentration_evolution.csv.")
enregistrer(fig, "4_concentration.png")

ecrit 4_concentration.png


## 5. La corrélation entre les titres du thème

In [6]:
glissante = pd.read_csv(DONNEES / "risque_correlation_glissante.csv", index_col=0, parse_dates=True).rho
fig, ax = plt.subplots(figsize=(10, 4.4))
tensions(ax)
ax.plot(glissante.index, glissante, color=THEME)
ax.axhline(glissante.median(), color=MARCHE, lw=1, ls="--")
ax.text(glissante.index[3], glissante.median() + 0.015, f"médiane {glissante.median():.2f}", color=ENCRE_2, fontsize=9)
ax.set_ylim(0, None)
ax.set_ylabel("Corrélation moyenne implicite, 252 séances")
ax.set_title("La corrélation monte avec la volatilité du marché")
ax.text(0, 1.01, "Après correction de Forbes et Rigobon, la hausse en crise disparaît : voir la section XI du rapport",
        transform=ax.transAxes, color=ENCRE_2, fontsize=10)
source(fig, "Titres de P1. Fond gris : épisodes de tension. Source : data/processed/risque_correlation_glissante.csv.")
enregistrer(fig, "5_correlation.png")

ecrit 5_correlation.png


## 6. Ce que le hasard permet : les écarts de Sharpe entre les deux gestions

In [7]:
boot = pd.read_csv(DONNEES / "risque_bootstrap_gestion.csv")
b = boot[boot.mesure == "sharpe"].iloc[::-1]
y = np.arange(len(b))
fig, ax = plt.subplots(figsize=(10, 4.8))
couleurs = [THEME if s else MARCHE for s in b.significatif_5pct]
ax.hlines(y, b.bas, b.haut, color=couleurs, lw=2)
ax.scatter(b.ecart, y, s=60, color=couleurs, zorder=3, edgecolor=FOND, linewidth=2)
ax.axvline(0, color=ENCRE_2, lw=1)
ax.set_yticks(y, b.portefeuille)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Sharpe rééquilibré moins Sharpe conservé, intervalle à 95 %")
ax.set_title("Neuf écarts de Sharpe sur dix sont compatibles avec le hasard")
ax.text(0, 1.01, "Bootstrap par blocs d'un trimestre, 2 000 tirages ; en bleu, écart significatif à 5 %",
        transform=ax.transAxes, color=ENCRE_2, fontsize=10)
source(fig, "Source : data/processed/risque_bootstrap_gestion.csv, produit par src/robustesse_statistique.ipynb.")
enregistrer(fig, "6_bootstrap_sharpe.png")

ecrit 6_bootstrap_sharpe.png
